In [ ]:
import search as search 

import time
import matplotlib.pyplot as plt
import numpy as np


# Creating an AI Agent to Solve a Maze!

In this tutorial, we're going to use depth first search (DFS) and breadth first search (BFS) to allow an agent to solve the path through a maze.

We will build off the search.py file and the Problem class in this file to build the search problem. **This content is directly applicable to Project 2, and during the tutorial we will discuss the common links.**

First, let's go to search.py and look through how this module works to allow you to implement and solve search problems. Open search.py and try to gain an initial understanding of how these classes and functions work:

* depth_first_graph_search() and breadth_first_graph_search()
* graph_search()
* class Node()
* class Problem()


Now we're going to use these functions and classes to build a maze solver! The first maze we will solve is shown below -- first visually, and then as a string. Each maze we will solve has a single start 'S' and single goal 'G'. Walls are indicated with '#'. 

![alt text](Simple_Maze.png)

**Consider: How is the string below similar or different to the warehouse representation in Project 2?**

In [ ]:
maze1 = """########
#      #
# #### #
#    # #
# ## # #
#  #   #
#S##G###"""

## Using classes to store information about our world state

Our string representation above is intuitive to type out to draw a maze, but it's not very useful for an algorithm to use. Read through the class Maze below, which given any maze string will process the string to find the walls, goal, and start, and store all their locations as (x,y) coordinates.

**Consider: How is this similar or different to the Warehouse class in sokoban.py for Project 2?**

In [ ]:
class Maze:
    '''
    Given a string of a maze, with # for walls, S for a starting position, and G for the goal,
    this class will extract the (x,y) location of walls, (x,y) location of the start,
    and (x,y) location of the goal.
    You can also use this class to print the maze.
    '''
    def __init__(self, maze):

        self.maze = maze
        
        self.walls = []
        self.find_walls(maze)

        self.goal = None
        self.find_goal(maze)

        self.start = None
        self.find_start(maze)

    def find_walls(self, maze):
        #store the (x,y) coordinates of all the walls in the maze
        rows = maze.split('\n')
        for y, line in enumerate(rows):
            self.walls += [(x, y) for x, letter in enumerate(line) if letter == '#']

    def find_goal(self, maze):
        #search for a 'G' and return position
        rows = maze.split('\n')
        for y, line in enumerate(rows):
            for x, char in enumerate(line):
                if char == 'G':
                    self.goal = (x, y)
                    return

        if self.goal == None:
            print('Error, no goal found.')

    def find_start(self, maze):
        #search for a 'S' and return position
        rows = maze.split('\n')
        for y, line in enumerate(rows):
            for x, char in enumerate(line):
                if char == 'S':
                    self.start = (x, y)
                    return (x,y)

        if self.start == None:
            print('Error, no start found.')
                                        
    
    def print(self, m = None):
        #can print the string of any maze passed in, otherwise prints the stored maze
        if m == None:
            m = self.maze
        for line in m.split('\n'):
            print(''.join(line))

    def visualise(self, maze = None):
        #can print the string of any maze passed in, otherwise prints the stored maze
        if maze == None:
            maze = self.maze
            
        # Convert the maze string into a list of lists
        maze_rows = maze.split('\n')
        height = len(maze_rows)
        width = len(maze_rows[0])

        # Create a 2D numpy array to store the maze representation
        maze_array = np.ones((height, width))
        
        # Fill the numpy array: 1 for walls ('#')
        for y, row in enumerate(maze_rows):
            for x, char in enumerate(row):
                if char == '#':
                    maze_array[y, x] = 0  # Wall
                elif char == 'o':
                    maze_array[y, x] = 0.5  # Optional: Distinguish path as gray
        
        # Plot the maze using matplotlib
        plt.figure(figsize=(2, 2))
        plt.imshow(maze_array, cmap='gray')
        plt.axis('off')  # Hide the axis
        plt.title("Maze Visualization")
        plt.show()


Below, we can use our Maze class with our maze1 string to create a Maze instance. Try printing some of the fields (e.g. walls, start, goal) and comparing it with the maze visualisation.

In [ ]:
maze = Maze(maze1)


## Creating a Maze Solver!

Below you can see the Maze_Solver class which is a subclass of the search.Problem class. Read through, noting what has already been implemented for you, and which methods have not been implemented.

In [ ]:
class Maze_Solver(search.Problem):
    '''
    An instance of the class 'SokobanPuzzle' represents a Sokoban puzzle.
    An instance contains information about the walls, the targets, the boxes
    and the worker.

    Your implementation should be fully compatible with the search functions of 
    the provided module 'search.py'. 
    '''
    def __init__(self,
                 maze,
                 ): 

        self.maze = maze
        self.goal = maze.goal 
        self.initial = maze.start 
       
        self.initial = tuple(self.initial)# use tuple to make the state hashable
        self.goal = tuple(self.goal) # use tuple to make the state hashable
    ## - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
    

    def actions(self, state):
        """
        Return the list of actions that can be executed in the given state.
        """
        raise NotImplementedError()


    def result(self, state, action):
        """
        Return the state that results from executing the given
        action in the given state. The action must be one of
        self.actions(state).

        The updated state should be returned as a tuple, i.e. return tuple(state), to make the state hashable
        """
        raise NotImplementedError()
        
    
    def print_solution(self, goal_node):
        """
            Shows solution represented by a specific goal node.
            For example, goal node could be obtained by calling 
                goal_node = breadth_first_tree_search(problem)
        """
        # path is list of nodes from initial state (root of the tree)
        # to the goal_node
        path = goal_node.path()
        # print the solution
        print( f"Solution takes {len(path)-1} steps from the initial state to the goal state\n")
        print( "Below is the sequence of moves\n")
        moves = []
        for node in path:
            if node.action:
                moves += [f"{node.action}, "]
        print(moves)
            
        print( "Below is the path drawn on the maze\n")
        self.print_maze_solution(path)
        

    def print_maze_solution(self, path):
        s = self.maze.maze

        maze_rows = [list(row) for row in s.split('\n')]

        for node in path:
            x,y = node.state
            maze_rows[y][x] = 'o'
        
        s_path = '\n'.join([''.join(row) for row in maze_rows])
        self.maze.visualise(s_path)               


Below we are creating an instance of the Maze_Solver that uses our maze object built from maze1.

Try printing the start and the goal.

**Consider: What do you think the state we are going to change is for this problem? How does this differ to Project 2?**

In [ ]:
solver = Maze_Solver(maze)

## Using Search to Solve the search.Problem class

Below, you can call breadth_first_graph_search and depth_first_graph_search on your Maze_Solver! First, you will need to go back and complete the necessary methods in the Maze_Solver class.

Once you have done this, compare BFS and DFS and their solutions, as well as the number of steps they used and the time taken to find a solution.

In [ ]:
search_type = 'bfs'

t0 = time.time()
if search_type == 'bfs':
    # Solve with Breadth First Search
    sol_ts = search.breadth_first_graph_search(solver)
else:
    #Solve with Depth First Search
    sol_ts = search.depth_first_graph_search(solver)
t1 = time.time()

print (f"Solver took {1000*(t1-t0):.2f} milli-seconds to find a solution.")
solver.print_solution(sol_ts)

## More complex Search Problems
Below, we have a much larger and more difficult maze! 


In [ ]:
maze2 = '''#S###################
#       #       #   #
### # # ### ### # ###
#   # #     # # #   #
# ### ####### # ### #
#     #             #
### ### # ########  #
#           # #     #
# ### # ### # ### # #
# #   #     #   # # #
# # ####### ### # # #
#           # # # # #
# ####### ### # # # #
#       #   #       #
## #### ######### ###
#   #             # #
# # # ### # ##### # #
#     #     #   #   #
#################G###'''

maze = Maze(maze2)
maze.visualise()

Build the cell that can take this new maze to build a Maze_Solver and again run DFS and BFS. Compare their solutions, time taken, and number of steps. How does this compare to what you learnt about these search types in the lectures?

## Concluding Remarks

**Consider: How is this relevant to Project 2? It may be worth opening your assignment and MySokobanSolver.py to note the similarities and differences with what we have done in this tutorial.**

In any time remaining, try building your own maze and observing how search can find a solution.

**Consider: what happens when you pass in a maze with no goal?**

In [ ]:
maze_impossible = """########
#      #
# #### #
#    # #
# ######
#  #   #
#S##G###"""

maze = Maze(maze_impossible)
maze.visualise()